# Multi-Modal E-commerce Recommendation Engine

In [11]:
import pandas as pd

articles = pd.read_csv("../data/articles.csv")
print(len(articles))
print(articles.columns.to_list())
articles.head() 

105542
['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [10]:
description_nulls = articles['detail_desc'].isnull().sum()
product_nulls = articles['prod_name'].isnull().sum()
types = articles['product_type_name'].nunique()

print(f"Nulls in detail descriptions: {description_nulls}")
print(f"Nulls in product names: {product_nulls}")
print(f"Number of types of products: {types}")

Nulls in detail descriptions: 416
Nulls in product names: 0
Number of types of products: 131


In [9]:
# dropping rows with missing values
articles_cleaned = articles.dropna(subset=['detail_desc']) 

# sampling evenly across all categories for a smaller subset of the data and so no one category dominates
sampled_groups = []
for product_type, group in articles_cleaned.groupby('product_type_name'):
    n = min(len(group), 9)
    sampled_groups.append(group.sample(n=n, random_state=42)) 

subset = pd.concat(sampled_groups, ignore_index=True)    

print(f"Sampled {len(subset)} products across {subset['product_type_name'].nunique()} categories")
subset[['article_id', 'prod_name', 'product_type_name', 'detail_desc']].head()

Sampled 970 products across 131 categories


,article_id,prod_name,product_type_name,detail_desc
0,755356001,RUBBER ANIMALS,Accessories set,"Small, glow-in-the-dark plastic toy dinosaurs ..."
1,858306002,Esther set 2pcs,Accessories set,Fleece-lined set with a hat and pair of mitten...
2,858306006,Esther set 2pcs,Accessories set,Fleece-lined set with a hat and pair of mitten...
3,858306003,Esther set 2pcs,Accessories set,Fleece-lined set with a hat and pair of mitten...
4,858306005,Esther set 2pcs,Accessories set,Fleece-lined set with a hat and pair of mitten...


In [ ]:
# staying consistent with the data's structure by adding 0s infront of the numbers
subset['article_id_str'] = "0" + subset['article_id'].astype(str)
subset['image_path'] = subset['article_id_str'].apply(lambda x: f"images/{x[:3]}/{x}.jpg")
subset[['article_id', 'article_id_str', 'image_path']]

,article_id,article_id_str,image_path
0,755356001,0755356001,images/075/0755356001.jpg
1,858306002,0858306002,images/085/0858306002.jpg
2,858306006,0858306006,images/085/0858306006.jpg
3,858306003,0858306003,images/085/0858306003.jpg
4,858306005,0858306005,images/085/0858306005.jpg
...,...,...,...
965,916926003,0916926003,images/091/0916926003.jpg
966,850239001,0850239001,images/085/0850239001.jpg
967,852389001,0852389001,images/085/0852389001.jpg
968,852390001,0852390001,images/085/0852390001.jpg


In [7]:
# downloading a subset of images matched with the articles
import subprocess
import os

os.makedirs("../data/images", exist_ok=True)

failed = []
for i, r in subset.iterrows():
    img_path = r['image_path']
    local_filename = r['article_id_str'] + ".jpg"
    local_path = f"../data/images/{local_filename}"
    if os.path.exists(local_path):
        continue
    result = subprocess.run(['kaggle', 'competitions', 'download', '-c', 'h-and-m-personalized-fashion-recommendations', '-f',
                             img_path, '-p', '../data/images', '--force'], capture_output=True, text=True)
    if result.returncode != 0:
        failed.append(img_path)

print(f"Done. {len(failed)} failed downloads.")
if failed:
    print(failed[:10])

Done. 6 failed downloads.
['images/056/0567547003.jpg', 'images/053/0532227002.jpg', 'images/051/0511653001.jpg', 'images/056/0563954001.jpg', 'images/053/0537732001.jpg', 'images/057/0570319011.jpg']


In [ ]:
# verifying downloads and creating a column for products that actually have images 
subset['image_exists'] = subset['article_id_str'].apply(lambda x: os.path.exists(f"../data/images/{x}.jpg"))
print(f"Products with images: {subset['image_exists'].sum()} / {len(subset)}")

subset_final = subset[subset["image_exists"]].drop(columns=['image_exists']).reset_index(drop=True)
print(f"Final dataset size: {len(subset_final)}") 

Products with images: 964 / 970
Final dataset size: 964


## Model for image embeddings: ResNet50
ResNet50 is a good model choice for this project because it is a well-trained convolutional network trained on millions of images on ImageNet. A pre-trained model means you don't have to train a vision model from scratch and gives you a general purpose visual feature extraction. 

In [ ]:
import torch
import torchvision.models
import torchvision.transforms as transforms
from PIL import Image
import random

# setting random states
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# 50 layer resnet with weights trained on imagenet
resnet = torchvision.models.resnet50(weights='IMAGENET1K_V2')
# taking all layers except the last one which is the classification layer - since we are only embedding, we don't need that - and
# reassmbling into a running model
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
# evaluation mode
resnet.eval()

# standard steps apllied to an image before it gets fed into the model to stay consistent with how resnet was trained with imagenet
preprocess = transforms.Compose([transforms.Resize(256),
                                 transforms.CenterCrop(224),
                                 transforms.ToTensor(),
                                 transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])])

# image embedding function that converts image into rgb scale, uses the preprocessing pipeline above, pads it, and runs through
# resnet without tracking gradients
def image_embeddings(image_path):
    img = Image.open(image_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        embedding = resnet(img_tensor)
    return embedding.squeeze().numpy()

test = f"../data/images/{subset_final['article_id_str'].iloc[0]}.jpg"
test_embeds = image_embeddings(test)
print(test_embeds.shape)


(2048,)


In [ ]:
from tqdm import tqdm

img_embeds_lst = []
failed = []
for i,r in tqdm(subset_final.iterrows(), total=len(subset_final)):
    img_path = f"../data/images/{r['article_id_str']}.jpg"
    # defensize coding to keep from crashing entire loop
    try:
        embed = image_embeddings(img_path)
        img_embeds_lst.append(embed)
    except Exception as e:
        failed.append((r['article_id_str'], str(e)))
        img_embeds_lst.append(None)

print(f"Embedded {len(img_embeds_lst) - len(failed)} successfully.")
print(f"{len(failed)} failed.")

100%|██████████| 964/964 [02:25<00:00,  6.61it/s]

Embedded 964 successfully.
0 failed.


## Model Choice for Text Embeddings: SentenceTransformer
SentenceTransformer is a fast and free model to generate embeddings for text. It does not require an API and produces good general-purpose embeddings. Combined product name and detail description to create better quality embeddings for the model. 

In [ ]:
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer("all-MiniLM-L6-v2")
subset_final['text_combined'] = subset_final['prod_name'] + '. ' + subset_final['detail_desc']
text_embeddings = text_model.encode(subset_final['text_combined'].tolist(), show_progress_bar=True, batch_size=32)

print(text_embeddings.shape)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 31/31 [00:09<00:00,  3.17it/s]

(964, 384)


The image embeddings and text embeddings have different dimensions and as a result, are not comparable. The solution is to combine them into a shared dimensional space. We build two small learned projection networks to map the embeddings to a shared dimensional space. This has the same idea as CLIP (contrastive-language image-pretraining). I specifically didn't use CLIP because it isn't domain-specific for this project. It is trained on broad, general data and its concept of 'similarity' is optimized for that, not for fashion/products specific vocabulary. 

I also wanted to implement this so I can challenge myself and develop an actual understanding of the underlying concepts on contrastive learning, and how to project different modalities. 

In [12]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 72.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 100.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'


# projection into shared dimensional space since text and image embeddings have different dimensions
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np

img_embeds = np.array(img_embeds_lst)
text_embeds = np.array(text_embeddings)

SHARED_DIM = 256

# projecting images from 2048 to 256 using keras Dense
img_input = layers.Input(shape=(2048,))
img_proj = layers.Dense(512, activation='relu')(img_input)
img_proj = layers.Dense(SHARED_DIM)(img_proj)
# normalizing to compare with cosine similarity
img_proj = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(img_proj)
img_encoder = Model(img_input, img_proj, name="img_encoder")

# projecting text from 384 to 256 using keras Dense
text_input = layers.Input(shape=(384,))
text_proj = layers.Dense(512, activation='relu')(text_input)
text_proj = layers.Dense(SHARED_DIM)(text_proj)
# normalizing to simplify the comparison with cosine similarity
text_proj = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(text_proj)
text_encoder = Model(text_input, text_proj, name="text_encoder")

img_encoder.summary()
text_encoder.summary()

I0000 00:00:1783189199.093171   22713 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783189200.779256   22713 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783189203.830195   22713 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783189203.831995   22713 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Model: "img_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,180,416 (4.50 MB)

 Trainable params: 1,180,416 (4.50 MB)

...

Model: "text_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 384)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,448 (1.25 MB)

 Trainable params: 328,448 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# contrastive loss function - teaches the model that an image and a text belong together
def contrastive_loss(img_embeds, text_embeds, temp=0.07):
    # computing similarity between every image and text in the batch
    logits = tf.matmul(img_embeds, text_embeds, transpose_b=True)/temp
    batch_length = tf.shape(img_embeds)[0]
    labels = tf.range(batch_length)
    loss_i = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
    loss_t = tf.keras.losses.sparse_categorical_crossentropy(labels, tf.transpose(logits), from_logits=True)
    res = tf.reduce_mean(loss_i + loss_t)/2
    return res

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
BATCH_SIZE=32
EPOCHS=20
n_samps = len(img_embeds)
idxs = np.arange(n_samps)
img_embeds_tensors = tf.constant(img_embeds, dtype=tf.float32)
txt_embeds_tensors = tf.constant(text_embeds, dtype=tf.float32)

# compiles func for faster repeated execution
@tf.function 
def training(img_batch, text_batch):
    # tracking operations to later compute gradient
    with tf.GradientTape() as tape:
        image_proj = img_encoder(img_batch, training=True)
        text_proj = text_encoder(text_batch, training=True)
        loss = contrastive_loss(image_proj, text_proj)
    # computing gradients for both encoders' trainable weights together
    trainable_variables = img_encoder.trainable_variables + text_encoder.trainable_variables
    # calculates how much each trainable weight contributed to the loss
    gradients = tape.gradient(loss, trainable_variables)
    # updates weights based on computed gradients using Adam algorithm
    optimizer.apply_gradients(zip(gradients, trainable_variables))
    return loss

hist = []
for epoch in range(EPOCHS):
    np.random.shuffle(idxs)
    epoch_losses = []
    for i in range(0, n_samps, BATCH_SIZE):
        batch_idx = idxs[i:i + BATCH_SIZE]
        img_batch = tf.gather(img_embeds_tensors, batch_idx)
        txt_batch = tf.gather(txt_embeds_tensors, batch_idx)
        loss = training(img_batch, txt_batch)
        epoch_losses.append(loss.numpy())

    avg_loss = np.mean(epoch_losses)
    hist.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - loss is {avg_loss:.4f}")



Epoch 1/20 - loss is 3.1836
Epoch 2/20 - loss is 2.1260
Epoch 3/20 - loss is 1.3631
Epoch 4/20 - loss is 0.8705
Epoch 5/20 - loss is 0.5931
Epoch 6/20 - loss is 0.4322
Epoch 7/20 - loss is 0.3482
Epoch 8/20 - loss is 0.2737
Epoch 9/20 - loss is 0.2313
Epoch 10/20 - loss is 0.1963
Epoch 11/20 - loss is 0.1812
Epoch 12/20 - loss is 0.1491
Epoch 13/20 - loss is 0.1368
Epoch 14/20 - loss is 0.1148
Epoch 15/20 - loss is 0.1173
Epoch 16/20 - loss is 0.1033
Epoch 17/20 - loss is 0.0937
Epoch 18/20 - loss is 0.0835
Epoch 19/20 - loss is 0.0843
Epoch 20/20 - loss is 0.0760


In [ ]:
# project all 964 products into shared space using trained encoders
final_img_embeds = img_encoder.predict(img_embeds_tensors, batch_size=32)
final_txt_embeds = text_encoder.predict(txt_embeds_tensors, batch_size=32)
# combine representations into one per product
combined_embeddings = (final_img_embeds + final_txt_embeds)/2
print(combined_embeddings.shape) 

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
(964, 256)


## Model Choice for Candidate Retrieval: BallTree
BallTree is a standard and efficient spatial data structure that organizes points hierarchically so similarity search doesn't need to check every single item. BallTree performs good in small scales, but will degrade at a very large scale. For a bigger scale, a proper nearest-neighbor library is appropriate (such as FAISS). 

In [ ]:
# now BallTree for candidate retrieval
from sklearn.neighbors import BallTree

tree = BallTree(combined_embeddings, metric='euclidean')

def recommendations_retrieval(prod_idx, k=5):
    query_embeds = combined_embeddings[prod_idx].reshape(1,-1)
    dist, idx = tree.query(query_embeds, k=k+1)
    # skipping first result becaise its always itself
    similar_idx = idx[0][1:]
    similar_dist = dist[0][1:]
    res = subset_final.iloc[similar_idx], similar_dist
    return res

In [ ]:
# testing the recommendations function
recommends, dists = recommendations_retrieval(0, k=5)
print(subset_final.iloc[0][['prod_name', 'product_type_name']])
print("Recommended similar products:")
print(recommends[['prod_name', 'product_type_name']])

prod_name             RUBBER ANIMALS
product_type_name    Accessories set
Name: 0, dtype: object
Recommended similar products:
              prod_name product_type_name
266      Lip balm Bunny    Fine cosmetics
720         Fur Friends         Soft Toys
717  ST Animal Princess         Soft Toys
716         Alma Onesie         Soft Toys
714   ST Fur friend BFF         Soft Toys


In [ ]:
# testing more for small evaluation
test_idx = [25, 60, 340, 875, 500, 644]

for i in test_idx:
    query = subset_final.iloc[i]
    recs, dists = recommendations_retrieval(i, k=5)
    print(f"Query: {query['prod_name']} ({query['product_type_name']})")
    for _, rec in recs.iterrows():
        print(f" - {rec['prod_name']} ({rec['product_type_name']})")

Query: Bag Simon Rolltop (Bag)
 - Jalle backpack (Backpack)
 - Larry crossbodu (Cross-body bag)
 - ELIZABETH small backpack (Backpack)
 - MINDY SHOPPER (Weekend/Gym bag)
 - MINDY SHOPPER (Weekend/Gym bag)
Query: Knot Bitter Top (Bikini top)
 - Locomotion Balconette (Bikini top)
 - Baby shark top (Bikini top)
 - Timeless Low V Shape Brief (Swimwear bottom)
 - CANDY Bikini (Swimwear set)
 - SIMPLE AS THAT BIKINI (Swimwear set)
Query: Small basic scrunchie (Hair ties)
 - Medium scrunchie (Hair clip)
 - H-string 3pk big scrunchy (Hair string)
 - Pu scrunchie (Hair ties)
 - 4p Bella scrunchies (Hair string)
 - Velvet scrunchie (Hair ties)
Query: PE TYRA BOTTOM (Underwear bottom)
 - BREE hipster bottom (Swimwear bottom)
 - OP  hipster reg mid 2p (Unknown)
 - OP Cheeky hipster 2p (Unknown)
 - Timeless Low V Shape Brief (Swimwear bottom)
 - Sweet & Bitter High Rise Brief (Swimwear bottom)
Query: Flirty Cora Pouch (Other accessories)
 - W Christie zip around wallet (Wallet)
 - Flirty Melon Mini